# SHS Replication — Full Analysis (Martell & Rodewald, 2024)

This notebook replicates the analyses of Martell & Rodewald (2024) on the EPFL European sample.

**Pipeline**
1. Load data, exclude failed attention checks, clean Likert items, build composite indices.
2. Scale reliability (Cronbach's α) for the 5 indices.
3. Sample descriptives by condition.
4. **H1** — Effect of frame on positive & negative affect (experimental conditions only).
5. **H2** — Pessimistic vs Optimistic on donation intentions.
6. **H3** — Combined frame vs single-frame on behavioral intentions.
7. **H4** — Mediation: frame → affect → behavioral intention (PROCESS-style bootstrap, 10 000 iterations).
8. **Extension** — Donation-link click-through (χ² + logistic regression).
9. Summary table & exported plots.

Original analyses used SPSS v28 + Hayes PROCESS macro. Here we use `statsmodels`, `scipy`, `scikit-learn`, with bootstrap mediation implemented manually. All α = 0.05; pairwise contrasts use Bonferroni; CIs are 95 % bias-corrected.


In [ ]:
# Imports
import os, json, itertools, warnings
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
RNG = np.random.default_rng(42)
ALPHA = 0.05
N_BOOT = 10_000

OUT = "data/results"
os.makedirs(f"{OUT}/plots_html", exist_ok=True)
os.makedirs(f"{OUT}/plots_png", exist_ok=True)

CONDITION_ORDER = ["Pessimistic", "Optimistic", "Combined", "Control"]
CONDITION_COLORS = {"Pessimistic": "#d62728", "Optimistic": "#2ca02c",
                    "Combined": "#1f77b4", "Control": "#7f7f7f"}

try:
    import kaleido  # noqa: F401
    KALEIDO = True
except ImportError:
    KALEIDO = False

def show(fig, name, w=950, h=550):
    fig.update_layout(template="plotly_white", width=w, height=h)
    fig.write_html(f"{OUT}/plots_html/{name}.html", include_plotlyjs="cdn")
    if KALEIDO:
        try:
            fig.write_image(f"{OUT}/plots_png/{name}.png", scale=2)
        except Exception as e:
            print(f"PNG export skipped: {e}")
    fig.show()

print("Setup ready.")

Setup ready.


## 1. Load, clean, build indices

In [2]:
raw = pd.read_csv("data/SHS_raw.csv")
print("Raw rows:", len(raw))

df = raw.copy()
# Strip stray non-breaking spaces from column names
df.columns = [c.replace("\xa0", " ").strip() for c in df.columns]

# Keep finished & with valid condition
df = df[(df["Finished"] == True) & (df["Condition"].notna()) & (df["Condition"] != "NA")].copy()
print("After Finished+Condition filter:", len(df))

# Attention check exclusion (only experimental conditions saw it)
exp_mask = df["Condition"].isin(["Pessimistic", "Optimistic", "Combined"])
failed_attn = exp_mask & (df["Attention"].astype(str).str.strip().str.lower() != "birds")
print(f"Failed attention check (experimental): {failed_attn.sum()}")
df = df[~failed_attn].copy()
print("Final analytic N:", len(df))

def clean_likert(s):
    return s.astype(str).str.extract(r"(\d+)", expand=False).astype(float)

ACTION_MAP = {"Extremely likely": 5, "Somewhat likely": 4,
              "Neither likely nor unlikely": 3,
              "Somewhat unlikely": 2, "Extremely unlikely": 1}

# Locate item columns
neg_cols = sorted([c for c in df.columns if c.startswith("EmotionNegative")])
pos_cols = sorted([c for c in df.columns if c.startswith("EmotionPositive")])
action_cols = sorted([c for c in df.columns if c.startswith("Actions")])
polit_cols  = sorted([c for c in df.columns if c.startswith("PoliticalActions")])

print("Neg emotion items :", neg_cols)
print("Pos emotion items :", pos_cols)
print("Pro-conservation  :", action_cols)
print("Political eng.    :", polit_cols)

# Convert Likert text → numeric
for c in neg_cols + pos_cols:
    df[c + "_n"] = clean_likert(df[c])
for c in action_cols + polit_cols:
    df[c + "_n"] = df[c].map(ACTION_MAP)

# Composite indices
df["NegAffect"] = df[[c + "_n" for c in neg_cols]].mean(axis=1)
df["PosAffect"] = df[[c + "_n" for c in pos_cols]].mean(axis=1)
df["ProCons"]   = df[[c + "_n" for c in action_cols]].mean(axis=1)
df["PolEng"]    = df[[c + "_n" for c in polit_cols]].mean(axis=1)

# Donation intentions: yes/no → 1/0, mean of three items
for c in ["DonationBird", "DonationWild", "DonationBio"]:
    df[c + "_b"] = (df[c].astype(str).str.strip().str.lower() == "yes").astype(int)
df["WTD"] = df[["DonationBird_b", "DonationWild_b", "DonationBio_b"]].mean(axis=1)
df["Donated_Any"] = df[["DonationBird_b", "DonationWild_b", "DonationBio_b"]].max(axis=1)

# Click-through (extension)
df["Clicked"] = pd.to_numeric(df["__js_button_clicked"], errors="coerce").fillna(0).astype(int)

df["Condition"] = pd.Categorical(df["Condition"], categories=CONDITION_ORDER, ordered=False)

df.to_csv(f"{OUT}/SHS_clean.csv", index=False)
df[["Condition","NegAffect","PosAffect","ProCons","PolEng","WTD","Donated_Any","Clicked"]].head()

Raw rows: 110
After Finished+Condition filter: 77
Failed attention check (experimental): 1
Final analytic N: 76
Neg emotion items : ['EmotionNegative _1', 'EmotionNegative _2', 'EmotionNegative _3', 'EmotionNegative _4']
Pos emotion items : ['EmotionPositive _1', 'EmotionPositive _2', 'EmotionPositive _3']
Pro-conservation  : ['Actions _1', 'Actions _2', 'Actions _3', 'Actions _4', 'Actions _5', 'Actions _6', 'Actions _7']
Political eng.    : ['PoliticalActions _1', 'PoliticalActions _2', 'PoliticalActions _3', 'PoliticalActions _4', 'PoliticalActions _5', 'PoliticalActions _6', 'PoliticalActions _7']


,Condition,NegAffect,PosAffect,ProCons,PolEng,WTD,Donated_Any,Clicked
0,Pessimistic,3.75,1.333333,3.857143,3.000000,1.0,1,0
1,Combined,1.00,1.000000,4.000000,2.714286,1.0,1,0
2,Optimistic,1.00,1.333333,3.714286,1.857143,1.0,1,0
3,Control,NaN,NaN,2.857143,1.000000,0.0,0,0
4,Optimistic,2.50,3.666667,3.571429,2.714286,1.0,1,0


## 2. Scale reliability (Cronbach's α)

In [3]:
def cronbach_alpha(items_df):
    items = items_df.dropna()
    k = items.shape[1]
    if k < 2 or len(items) < 2:
        return np.nan
    var_items = items.var(axis=0, ddof=1).sum()
    var_total = items.sum(axis=1).var(ddof=1)
    if var_total == 0:
        return np.nan
    return (k / (k - 1)) * (1 - var_items / var_total)

scales = {
    "Negative Affect (4 items)":      [c + "_n" for c in neg_cols],
    "Positive Affect (3 items)":      [c + "_n" for c in pos_cols],
    "Pro-conservation (7 items)":     [c + "_n" for c in action_cols],
    "Political engagement (7 items)": [c + "_n" for c in polit_cols],
    "WTD (3 items, binary)":          ["DonationBird_b", "DonationWild_b", "DonationBio_b"],
}

rel = pd.DataFrame({
    "Scale": list(scales.keys()),
    "Cronbach α (this sample)": [round(cronbach_alpha(df[cs]), 3) for cs in scales.values()],
    "α (original)": [0.887, 0.853, 0.854, 0.944, 0.843],
})
rel

,Scale,Cronbach α (this sample),α (original)
0,Negative Affect (4 items),0.836,0.887
1,Positive Affect (3 items),0.914,0.853
2,Pro-conservation (7 items),0.668,0.854
3,Political engagement (7 items),0.877,0.944
4,"WTD (3 items, binary)",0.795,0.843


## 3. Sample descriptives by condition

In [4]:
dvs = ["NegAffect", "PosAffect", "ProCons", "PolEng", "WTD"]
desc = (df.groupby("Condition", observed=True)[dvs]
          .agg(["mean", "std", "count"]).round(3))
desc

NegAffect              PosAffect              ProCons         \
                 mean    std count      mean    std count    mean    std   
Condition                                                                  
Pessimistic     2.431  0.844    18     1.574  0.694    18   3.270  0.715   
Optimistic      1.653  0.718    18     3.222  1.183    18   3.659  0.559   
Combined        2.275  0.734    20     2.333  0.898    20   3.429  0.827   
Control           NaN    NaN     0       NaN    NaN     0   3.629  0.641   

                  PolEng                 WTD               
            count   mean    std count   mean    std count  
Condition                                                  
Pessimistic    18  2.278  0.849    18  0.796  0.326    18  
Optimistic     18  2.365  0.888    18  0.667  0.379    18  
Combined       20  2.229  0.983    20  0.633  0.431    20  
Control        20  2.586  0.899    20  0.733  0.352    20

In [5]:
# Sample composition plot
counts = df["Condition"].value_counts().reindex(CONDITION_ORDER).reset_index()
counts.columns = ["Condition", "n"]
fig = px.bar(counts, x="Condition", y="n", color="Condition",
             color_discrete_map=CONDITION_COLORS, text="n",
             title="Sample size by condition (after exclusions)")
fig.update_traces(textposition="outside")
show(fig, "00_sample_by_condition", w=750, h=450)

## 4. H1 — Effect of frame on emotions

> Pessimistic > Optimistic on **Negative Affect**; Optimistic > Pessimistic on **Positive Affect**.
> Tested only on the three experimental conditions (control did not see emotion items).

Linear regression with demographic covariates (Age, Gender, Education, Political affiliation), backward stepwise on covariates (drop p > .10). Pairwise contrasts use Bonferroni.

In [6]:
def backward_stepwise(df_in, dv, focal, covariates, p_drop=0.10):
    # Backward stepwise on covariates; always keeps the focal predictor.
    keep = list(covariates)
    while keep:
        terms = [focal] + [f"C({c})" if df_in[c].dtype == object else c for c in keep]
        rhs = " + ".join(terms)
        model = smf.ols(f"{dv} ~ {rhs}", data=df_in).fit()
        pvals = model.pvalues
        worst, worst_p = None, -1
        for c in keep:
            related = [p for n, p in pvals.items() if n.startswith(f"C({c})") or n == c]
            if related:
                m = max(related)
                if m > worst_p:
                    worst, worst_p = c, m
        if worst is not None and worst_p > p_drop:
            keep.remove(worst)
        else:
            break
    if keep:
        terms = [focal] + [f"C({c})" if df_in[c].dtype == object else c for c in keep]
        rhs = " + ".join(terms)
    else:
        rhs = focal
    return smf.ols(f"{dv} ~ {rhs}", data=df_in).fit(), keep

def pairwise_bonferroni(df_in, dv, group_col="Condition", groups=None):
    groups = groups or [g for g in df_in[group_col].dropna().unique()]
    pairs = list(itertools.combinations(groups, 2))
    rows = []
    for a, b in pairs:
        x = df_in.loc[df_in[group_col] == a, dv].dropna()
        y = df_in.loc[df_in[group_col] == b, dv].dropna()
        if len(x) < 2 or len(y) < 2:
            continue
        t, p = stats.ttest_ind(x, y, equal_var=False)
        sd_pool = np.sqrt(((x.var(ddof=1)*(len(x)-1)) + (y.var(ddof=1)*(len(y)-1))) / (len(x)+len(y)-2))
        d = (x.mean() - y.mean()) / sd_pool if sd_pool > 0 else np.nan
        rows.append([a, b, len(x), len(y), x.mean()-y.mean(), t, p, p*len(pairs), d])
    out = pd.DataFrame(rows, columns=["A","B","nA","nB","meanA-meanB","t","p","p_bonf","d"])
    out["p_bonf"] = out["p_bonf"].clip(upper=1.0)
    return out.round(4)

COVARS = ["Age", "Gender", "EduLevel", "PoliticalAffi"]
df_exp = df[df["Condition"].isin(["Pessimistic", "Optimistic", "Combined"])].copy()
df_exp["Condition"] = df_exp["Condition"].astype(str)

print("=== H1a: Negative Affect ===")
m1a, kept1a = backward_stepwise(df_exp, "NegAffect",
                                "C(Condition, Treatment(reference='Optimistic'))", COVARS)
print("Retained covariates:", kept1a)
print(m1a.summary().tables[1])
print(pairwise_bonferroni(df_exp, "NegAffect"))

print("\n=== H1b: Positive Affect ===")
m1b, kept1b = backward_stepwise(df_exp, "PosAffect",
                                "C(Condition, Treatment(reference='Optimistic'))", COVARS)
print("Retained covariates:", kept1b)
print(m1b.summary().tables[1])
print(pairwise_bonferroni(df_exp, "PosAffect"))

=== H1a: Negative Affect ===
Retained covariates: []
                                                                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------------------
Intercept                                                          1.6528      0.181      9.153      0.000       1.291       2.015
C(Condition, Treatment(reference='Optimistic'))[T.Combined]        0.6222      0.249      2.500      0.016       0.123       1.121
C(Condition, Treatment(reference='Optimistic'))[T.Pessimistic]     0.7778      0.255      3.046      0.004       0.266       1.290
             A           B  nA  nB  meanA-meanB       t       p  p_bonf  \
0  Pessimistic    Combined  18  20       0.1556  0.6033  0.5503  1.0000   
1  Pessimistic  Optimistic  18  18       0.7778  2.9783  0.0054  0.0162   
2     Combined  Optimistic  20  18       0.6222  2.6390  0.0122  0.

In [7]:
# H1 visualization
def mean_ci(s, conf=0.95):
    s = s.dropna()
    if len(s) < 2: return s.mean(), 0
    se = s.std(ddof=1) / np.sqrt(len(s))
    h = se * stats.t.ppf((1 + conf) / 2, len(s) - 1)
    return s.mean(), h

emo_rows = []
for cond in ["Pessimistic", "Optimistic", "Combined"]:
    sub = df[df["Condition"] == cond]
    for dv, label in [("NegAffect", "Negative Affect"), ("PosAffect", "Positive Affect")]:
        m, h = mean_ci(sub[dv])
        emo_rows.append([cond, label, m, h])
emo_df = pd.DataFrame(emo_rows, columns=["Condition", "Affect", "Mean", "CI95"])

fig = px.bar(emo_df, x="Condition", y="Mean", color="Affect", barmode="group",
             error_y="CI95",
             title="H1 — Affect means (±95% CI) by experimental condition",
             color_discrete_sequence=["#d62728", "#2ca02c"])
fig.update_yaxes(range=[1, 5], title="Mean Likert score")
show(fig, "01_h1_affect")

## 5. H2 — Pessimistic vs Optimistic on donation intentions

> Pessimistic frame → higher donation intentions than optimistic frame.

Linear regression on **WTD** with covariates + Bonferroni pairwise contrasts. Also a logistic regression on `Donated_Any`.

In [8]:
print("=== H2: WTD ~ Condition + covariates (all 4 conditions) ===")
m2, kept2 = backward_stepwise(df, "WTD",
                              "C(Condition, Treatment(reference='Optimistic'))", COVARS)
print("Retained covariates:", kept2)
print(m2.summary().tables[1])
pw_wtd = pairwise_bonferroni(df, "WTD")
print(pw_wtd)

print("\n=== H2 (logistic): P(any donation) ~ Condition ===")
df_log = df.dropna(subset=["Donated_Any"]).copy()
logit = smf.logit("Donated_Any ~ C(Condition, Treatment(reference='Optimistic'))",
                  data=df_log).fit(disp=0)
print(logit.summary().tables[1])
print("\nOdds ratios:")
print(np.exp(logit.params).round(3))

=== H2: WTD ~ Condition + covariates (all 4 conditions) ===
Retained covariates: ['Age']
                                                                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------------------
Intercept                                                          0.8648      0.143      6.053      0.000       0.580       1.150
C(Condition, Treatment(reference='Optimistic'))[T.Pessimistic]     0.1001      0.124      0.804      0.424      -0.148       0.348
C(Condition, Treatment(reference='Optimistic'))[T.Combined]       -0.0669      0.122     -0.550      0.584      -0.310       0.176
C(Condition, Treatment(reference='Optimistic'))[T.Control]         0.0511      0.121      0.424      0.673      -0.189       0.291
Age                                                               -0.0066      0.004     -1.751      0.084      -0.014       

In [9]:
# H2 visualization
don_rows = []
for cond in CONDITION_ORDER:
    sub = df[df["Condition"] == cond]
    for col, lbl in [("DonationBird_b", "Birds"), ("DonationWild_b", "Wildlife"),
                     ("DonationBio_b", "Biodiversity"), ("WTD", "WTD index")]:
        m, h = mean_ci(sub[col])
        don_rows.append([cond, lbl, m, h])
ddf = pd.DataFrame(don_rows, columns=["Condition", "Item", "Rate", "CI95"])

fig = px.bar(ddf, x="Condition", y="Rate", color="Item", barmode="group", error_y="CI95",
             category_orders={"Condition": CONDITION_ORDER},
             title="H2 — Donation intentions by condition (±95% CI)")
fig.update_yaxes(range=[0, 1.05], title="Proportion / mean")
show(fig, "02_h2_donations")

## 6. H3 — Combined frame vs single-frame on behavioral intentions

> The combined condition will **not** produce higher behavioral intentions than the single-frame conditions.

Linear regressions on each behavioral DV with **Combined** as the reference category.

In [10]:
print("=== H3: behavioral DVs vs Combined reference ===")
h3 = {}
for dv, label in [("ProCons", "Pro-conservation behavior"),
                  ("PolEng",  "Political engagement"),
                  ("WTD",     "Willingness to donate")]:
    print(f"\n--- {label} ---")
    mod, kept = backward_stepwise(df, dv,
                                  "C(Condition, Treatment(reference='Combined'))", COVARS)
    print("Retained covariates:", kept)
    print(mod.summary().tables[1])
    h3[dv] = mod

=== H3: behavioral DVs vs Combined reference ===

--- Pro-conservation behavior ---
Retained covariates: []
                                                                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------------------
Intercept                                                        3.4286      0.155     22.057      0.000       3.119       3.738
C(Condition, Treatment(reference='Combined'))[T.Pessimistic]    -0.1587      0.226     -0.703      0.484      -0.609       0.291
C(Condition, Treatment(reference='Combined'))[T.Optimistic]      0.2302      0.226      1.019      0.312      -0.220       0.680
C(Condition, Treatment(reference='Combined'))[T.Control]         0.2000      0.220      0.910      0.366      -0.238       0.638

--- Political engagement ---
Retained covariates: []
                                                                

In [11]:
# H3 visualization
beh_rows = []
for cond in CONDITION_ORDER:
    sub = df[df["Condition"] == cond]
    for dv, lbl in [("ProCons", "Pro-conservation"),
                    ("PolEng", "Political engagement"),
                    ("WTD", "WTD")]:
        m, h = mean_ci(sub[dv])
        beh_rows.append([cond, lbl, m, h])
bdf = pd.DataFrame(beh_rows, columns=["Condition", "DV", "Mean", "CI95"])

fig = px.bar(bdf, x="Condition", y="Mean", color="DV", barmode="group", error_y="CI95",
             category_orders={"Condition": CONDITION_ORDER},
             title="H3 — Behavioral intentions by condition (±95% CI)")
fig.update_yaxes(title="Mean (Likert 1–5 / proportion for WTD)")
show(fig, "03_h3_behavior")

## 7. H4 — Mediation: frame → affect → behavioral intention

PROCESS Model 4 (Hayes): test whether **negative** and **positive** affect mediate the effect of frame on each behavioral DV. Implemented as a bootstrap (10 000 iterations) of the indirect effect a×b with percentile 95% CI. Restricted to experimental conditions; we contrast Pessimistic (1) vs Optimistic (0).

In [12]:
def boot_mediation(d, x, m, y, n_boot=N_BOOT, rng=RNG):
    d = d[[x, m, y]].dropna().reset_index(drop=True)
    n = len(d)
    if n < 10:
        return None
    a = sm.OLS(d[m], sm.add_constant(d[[x]])).fit().params[x]
    b = sm.OLS(d[y], sm.add_constant(d[[x, m]])).fit().params[m]
    c_total = sm.OLS(d[y], sm.add_constant(d[[x]])).fit().params[x]
    cprime = sm.OLS(d[y], sm.add_constant(d[[x, m]])).fit().params[x]
    indirect = a * b

    idx = np.arange(n)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        s = rng.choice(idx, size=n, replace=True)
        ds = d.iloc[s]
        ai = sm.OLS(ds[m], sm.add_constant(ds[[x]])).fit().params[x]
        bi = sm.OLS(ds[y], sm.add_constant(ds[[x, m]])).fit().params[m]
        boots[i] = ai * bi
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return dict(a=a, b=b, c=c_total, c_prime=cprime, indirect=indirect,
                CI95=(lo, hi), sig=(lo > 0) or (hi < 0))

med_df = df_exp[df_exp["Condition"].isin(["Pessimistic", "Optimistic"])].copy()
med_df["Pess"] = (med_df["Condition"] == "Pessimistic").astype(int)

results_h4 = {}
for dv, label in [("ProCons", "Pro-conservation"),
                  ("PolEng",  "Political engagement"),
                  ("WTD",     "Willingness to donate")]:
    for med, mlabel in [("NegAffect", "Negative Affect"), ("PosAffect", "Positive Affect")]:
        r = boot_mediation(med_df, "Pess", med, dv)
        if r is None:
            continue
        key = f"{label} via {mlabel}"
        results_h4[key] = r
        print(f"\n--- {key} ---")
        print(f"a (X→M)        = {r['a']:.3f}")
        print(f"b (M→Y|X)      = {r['b']:.3f}")
        print(f"c  (total)     = {r['c']:.3f}")
        print(f"c' (direct)    = {r['c_prime']:.3f}")
        print(f"indirect (a*b) = {r['indirect']:.3f}, 95% CI [{r['CI95'][0]:.3f}, {r['CI95'][1]:.3f}], significant = {r['sig']}")


--- Pro-conservation via Negative Affect ---
a (X→M)        = 0.778
b (M→Y|X)      = 0.328
c  (total)     = -0.389
c' (direct)    = -0.644
indirect (a*b) = 0.255, 95% CI [0.037, 0.642], significant = True

--- Pro-conservation via Positive Affect ---
a (X→M)        = -1.648
b (M→Y|X)      = 0.140
c  (total)     = -0.389
c' (direct)    = -0.158
indirect (a*b) = -0.231, 95% CI [-0.620, 0.182], significant = False

--- Political engagement via Negative Affect ---
a (X→M)        = 0.778
b (M→Y|X)      = 0.580
c  (total)     = -0.087
c' (direct)    = -0.538
indirect (a*b) = 0.451, 95% CI [0.107, 0.941], significant = True

--- Political engagement via Positive Affect ---
a (X→M)        = -1.648
b (M→Y|X)      = 0.356
c  (total)     = -0.087
c' (direct)    = 0.500
indirect (a*b) = -0.587, 95% CI [-1.189, -0.092], significant = True

--- Willingness to donate via Negative Affect ---
a (X→M)        = 0.778
b (M→Y|X)      = 0.088
c  (total)     = 0.130
c' (direct)    = 0.061
indirect (a*b) = 0

In [13]:
# H4 visualization
rows = []
for k, v in results_h4.items():
    rows.append([k, v["indirect"], v["CI95"][0], v["CI95"][1], v["sig"]])
mdf = pd.DataFrame(rows, columns=["Path", "Indirect", "lo", "hi", "sig"]).sort_values("Indirect")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=mdf["Indirect"], y=mdf["Path"], mode="markers",
    marker=dict(size=12, color=["#d62728" if s else "#7f7f7f" for s in mdf["sig"]]),
    error_x=dict(type="data", symmetric=False,
                 array=mdf["hi"] - mdf["Indirect"],
                 arrayminus=mdf["Indirect"] - mdf["lo"]),
    name="Indirect effect"))
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(title=("H4 — Bootstrapped indirect effects (a×b) with 95% CIs<br>"
                        "Pessimistic (1) vs Optimistic (0)"),
                  xaxis_title="Indirect effect (a×b)",
                  yaxis_title="")
show(fig, "04_h4_mediation", h=500)

## 8. Extension — Donation-link click-through

χ² test of click vs condition; logistic regression with affect indices and self-reported WTD as predictors.

In [14]:
ct = pd.crosstab(df["Condition"], df["Clicked"])
chi2, p_chi, dof, _ = stats.chi2_contingency(ct)
print("Click counts by condition:")
print(ct)
print(f"\nχ²({dof}) = {chi2:.3f}, p = {p_chi:.4f}")

rate = (df.groupby("Condition", observed=True)["Clicked"].mean()
          .reindex(CONDITION_ORDER).reset_index())
fig = px.bar(rate, x="Condition", y="Clicked", color="Condition",
             color_discrete_map=CONDITION_COLORS,
             text=rate["Clicked"].apply(lambda v: f"{v*100:.1f}%"),
             title="Donation-link click-through rate by condition")
fig.update_traces(textposition="outside")
ymax = max(0.05, rate["Clicked"].max()*1.4) if rate["Clicked"].max() > 0 else 0.05
fig.update_yaxes(range=[0, ymax], title="P(click)")
show(fig, "05_click_rate", h=450)

Click counts by condition:
Clicked       0  1
Condition         
Pessimistic  17  1
Optimistic   17  1
Combined     18  2
Control      19  1

χ²(3) = 0.523, p = 0.9137


In [15]:
print("=== Logistic: Clicked ~ WTD + Condition (all conditions) ===")
try:
    mfull = smf.logit("Clicked ~ WTD + C(Condition, Treatment(reference='Control'))",
                      data=df).fit(disp=0)
    print(mfull.summary().tables[1])
    print("Odds ratios:")
    print(np.exp(mfull.params).round(3))
except Exception as e:
    print("Logistic (full) failed:", e)

print("\n=== Logistic: Clicked ~ WTD + NegAffect + PosAffect (experimental only) ===")
try:
    mexp = smf.logit("Clicked ~ WTD + NegAffect + PosAffect", data=df_exp).fit(disp=0)
    print(mexp.summary().tables[1])
    print("Odds ratios:")
    print(np.exp(mexp.params).round(3))
except Exception as e:
    print("Logistic (experimental) failed:", e)

=== Logistic: Clicked ~ WTD + Condition (all conditions) ===
Logistic (full) failed: Singular matrix

=== Logistic: Clicked ~ WTD + NegAffect + PosAffect (experimental only) ===
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -53.4430   2113.728     -0.025      0.980   -4196.274    4089.388
WTD           39.7147   2113.719      0.019      0.985   -4103.098    4182.527
NegAffect      2.7015      1.505      1.795      0.073      -0.248       5.651
PosAffect      1.6877      1.104      1.529      0.126      -0.475       3.851
Odds ratios:
Intercept    0.000000e+00
WTD          1.769630e+17
NegAffect    1.490200e+01
PosAffect    5.407000e+00
dtype: float64


## 9. Summary

In [16]:
summary_rows = []

for label, mod in [("H1a NegAffect ~ Frame", m1a), ("H1b PosAffect ~ Frame", m1b)]:
    for n, v in mod.params.items():
        if "Condition" in n:
            summary_rows.append([label, n, round(v, 3), round(mod.pvalues[n], 4),
                                 "yes" if mod.pvalues[n] < 0.05 else ""])

for n, v in m2.params.items():
    if "Condition" in n:
        summary_rows.append(["H2 WTD ~ Frame", n, round(v, 3), round(m2.pvalues[n], 4),
                             "yes" if m2.pvalues[n] < 0.05 else ""])

for dv, mod in h3.items():
    for n, v in mod.params.items():
        if "Condition" in n:
            summary_rows.append([f"H3 {dv} ~ Frame (ref=Combined)", n, round(v, 3),
                                 round(mod.pvalues[n], 4),
                                 "yes" if mod.pvalues[n] < 0.05 else ""])

for k, v in results_h4.items():
    summary_rows.append([f"H4 {k}", "indirect a*b", round(v["indirect"], 3),
                         f"[{v['CI95'][0]:.3f}, {v['CI95'][1]:.3f}]",
                         "yes" if v["sig"] else ""])

summary_rows.append(["Ext chi2 Clicks~Cond", "chi2", round(chi2, 3), round(p_chi, 4),
                     "yes" if p_chi < 0.05 else ""])

summary = pd.DataFrame(summary_rows, columns=["Test", "Term", "Estimate", "p / 95% CI", "Sig"])
summary.to_csv(f"{OUT}/summary.csv", index=False)
summary

,Test,Term,Estimate,p / 95% CI,Sig
0,H1a NegAffect ~ Frame,"C(Condition, Treatment(reference='Optimistic')...",0.622,0.0156,yes
1,H1a NegAffect ~ Frame,"C(Condition, Treatment(reference='Optimistic')...",0.778,0.0036,yes
2,H1b PosAffect ~ Frame,"C(Condition, Treatment(reference='Optimistic')...",-0.889,0.0055,yes
3,H1b PosAffect ~ Frame,"C(Condition, Treatment(reference='Optimistic')...",-1.648,0.0,yes
4,H2 WTD ~ Frame,"C(Condition, Treatment(reference='Optimistic')...",0.100,0.4242,
5,H2 WTD ~ Frame,"C(Condition, Treatment(reference='Optimistic')...",-0.067,0.5842,
6,H2 WTD ~ Frame,"C(Condition, Treatment(reference='Optimistic')...",0.051,0.6728,
7,H3 ProCons ~ Frame (ref=Combined),"C(Condition, Treatment(reference='Combined'))[...",-0.159,0.4844,
8,H3 ProCons ~ Frame (ref=Combined),"C(Condition, Treatment(reference='Combined'))[...",0.230,0.3116,
9,H3 ProCons ~ Frame (ref=Combined),"C(Condition, Treatment(reference='Combined'))[...",0.200,0.366,


In [17]:
fig = go.Figure(data=[go.Table(
    header=dict(values=list(summary.columns), fill_color="#1f77b4",
                font=dict(color="white"), align="left"),
    cells=dict(values=[summary[c] for c in summary.columns],
               fill_color="#f7f7f7", align="left"))
])
fig.update_layout(title="Replication results — summary")
show(fig, "06_summary_table", h=min(900, 80 + 28 * len(summary)))